# **Query Translation**

For an in-depth discussion on the rationale behind Query Translation, including its specific use cases, methodological approaches, and typological distinctions, refer to the detailed documentation provided [here](what_is_query_translation.md).


##  Types of **Query Translation**
- Multi Query Translation
- HyDE (Hypothetical Document Embeddings)
- RAG Fusion
- Step-Back Reformulation
- Decomposition-Based Reformulation

| Technique         | Goal                        | When to Use            | Advantage                    | Limitation                   |
| ----------------- | --------------------------- | ---------------------- | ---------------------------- | ---------------------------- |
| **Multi-Query**   | Expand coverage             | Ambiguous queries      | Improves recall              | Higher compute cost          |
| **HyDE**          | Semantic enrichment         | Short or vague queries | Contextually rich retrieval  | Requires powerful generator  |
| **RAG Fusion**    | Combine multiple retrievals | Knowledge-heavy tasks  | Balanced precision-recall    | Complexity in fusion logic   |
| **Step-Back**     | Broaden context             | Narrow or deep queries | Enables high-level reasoning | Risk of diluting specificity |
| **Decomposition** | Structured reasoning        | Multi-hop queries      | Supports stepwise inference  | Integration complexity       |

### Multi Query Translation

- Loading and chunking the Documents

In [1]:
# Import necessary class
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Create a text splitter with defined chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 25,    # Each chunk will contain around 10 tokens
    chunk_overlap = 3   # 3 tokens of overlap between consecutive chunks
)

def load_documents():
    """
    Loads a small, diverse set of documents to simulate a knowledge base
    for Retrieval-Augmented Generation (RAG) experiments.

    Each document represents a single knowledge unit that will later be
    chunked, embedded, and indexed for similarity-based retrieval.

    Returns
    -------
    documents : list of langchain_core.documents.Document
        A list containing 10 textual documents across multiple domains.
    """

    # Define a diverse mini knowledge base covering various topics

    documents = [
        Document(page_content="Machine learning allows systems to automatically improve from experience without being explicitly programmed."),
        Document(page_content="Deep learning, a subset of machine learning, uses neural networks to model complex patterns in data."),
        Document(page_content="Neural networks are inspired by the structure of the human brain and form the foundation of deep learning."),
        Document(page_content="Python is a leading language for machine learning due to libraries like TensorFlow, PyTorch, NumPy, and Pandas."),
        Document(page_content="Data scientists use Python extensively for tasks like preprocessing, visualization, and training neural models."),
        Document(page_content="Artificial intelligence combines subfields like machine learning, deep learning, and natural language processing to mimic human intelligence."),
        Document(page_content="OpenAI’s GPT models leverage large datasets and transformer-based neural networks for human-like text generation."),
        Document(page_content="Retrieval-Augmented Generation (RAG) enhances language models by combining document retrieval with generation for factual accuracy."),
        Document(page_content="Large language models, such as GPT and Gemini, are trained using massive text corpora to understand and generate human-like responses."),
        Document(page_content="In modern AI pipelines, RAG systems use retrievers like BM25 and dense embeddings to fetch relevant information before generating answers.")
    ]


    # Log summary information for verification
    print(f"Total documents loaded: {len(documents)}")
    print("Displaying sample documents:")
    for i, doc in enumerate(documents[:3]):
        print(f"Document {i+1}: {doc.page_content[:100]}...")

    return documents
    
documents = load_documents()

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

# Display the created chunks
print(f"\nTotal chunks created: {len(chunks)}")
print("Displaying first 2 sample chunks:")
for i, chunk in enumerate(chunks[:2]):
    print(f"Chunk {i+1}: {chunk.page_content}")

Total documents loaded: 10
Displaying sample documents:
Document 1: Machine learning allows systems to automatically improve from experience without being explicitly pr...
Document 2: Deep learning, a subset of machine learning, uses neural networks to model complex patterns in data....
Document 3: Neural networks are inspired by the structure of the human brain and form the foundation of deep lea...

Total chunks created: 13
Displaying first 2 sample chunks:
Chunk 1: Machine learning allows systems to automatically improve from experience without being explicitly programmed.
Chunk 2: Deep learning, a subset of machine learning, uses neural networks to model complex patterns in data.


- Indexing the Documents

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer("all-MiniLM-L6-v2")
# Initialize an empty vector store to hold embeddings for each chunk
vector_store = []

# Convert each text chunk into a dense vector representation (embedding)
for idx, chunk in enumerate(chunks):
    text_content = chunk.page_content
    embedding_vector = model.encode(text_content)   # Convert text → vector
    vector_store.append(embedding_vector)

# Display the first 5 dimensions of first two chunk embeddings (for inspection)
[vector_store[0][:5], vector_store[1][:5]]

[array([-0.00684425,  0.0059964 ,  0.08135018,  0.06403239,  0.02934441],
       dtype=float32),
 array([-0.09046526, -0.02000159,  0.05460881, -0.00526585, -0.03021288],
       dtype=float32)]

- Multi-Query Translation

In [3]:
# Function to calculate cosine similarity between two vectors
def calculate_cosine_similarity_score_vect(v1, v2):
    dot_product = np.dot(v1, v2)
    v1_magnitude = np.linalg.norm(v1)
    v2_magnitude = np.linalg.norm(v2)
    cosine_similarity = dot_product / (v1_magnitude * v2_magnitude)
    return cosine_similarity

def retrieve_most_similar_chunk(model, chunks, vector_store, query_text):
    """
    Given a query, finds the most semantically similar chunk.
    Steps:
    1. Encode the query text into a vector.
    2. Compute cosine similarity with every chunk vector.
    3. Return the chunk with the highest similarity score.
    """
    encoded_query = model.encode(query_text)

    similarity_scores = []
    indexed_scores = []

    # Calculate similarity for each stored embedding
    for idx, chunk_vector in enumerate(vector_store):
        score = calculate_cosine_similarity_score_vect(encoded_query, chunk_vector)
        similarity_scores.append(score)
        indexed_scores.append((score, idx))

    # Log all similarity values for reference
    print("Calculated cosine similarity scores for each chunk:")
    for score, idx in indexed_scores:
        print(f"Score = {score:.4f}  |  Chunk Index = {idx}")

    # Retrieve the chunk with the maximum similarity score
    best_match_index = np.argmax(similarity_scores)
    best_chunk = chunks[best_match_index].page_content
    return best_chunk

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAI

# --------------------------------------------------------------------------
# Multi-Query Generation Model
# --------------------------------------------------------------------------
# Purpose:
# Generate multiple semantically diverse reformulations of the user's query.
# Each reformulated query represents a different perspective to improve
# document retrieval recall and reduce semantic bias in vector similarity search.

# Define the prompt template for the LLM
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 
    """
    You are an advanced language model. Your task is to generate five diverse reformulations 
    of the given user query to enhance document retrieval from a vector database.
    The objective is to capture multiple semantic perspectives of the original query,
    thereby mitigating the limitations of distance-based similarity search.
    Each reformulated query should be presented on a new line.
    """),
    ("user", "{user_query}")
])

# Initialize the generative LLM (Gemini 2.5-Flash)
llm = GoogleGenerativeAI(model="gemini-2.5-flash")

# Build the Multi-Query Generation Chain
# This chain takes the user query, sends it to the LLM, parses the output,
# and splits the generated reformulations line by line.
multi_query_chain = (prompt_template | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Example user query
query = "What are neural networks?"

# Generate multiple diverse reformulations for the given query
generated_queries = multi_query_chain.invoke(query)

print("Original Query:", query)
print("\nGenerated Queries by Multi-Query Generation Model:")
for idx, question in enumerate(generated_queries, start=1):
    print(f"Q.{idx}) {question}")

Original Query: What are neural networks?

Generated Queries by Multi-Query Generation Model:
Q.1) 1.  Please provide a detailed explanation of what neural networks are.
Q.2) 2.  How do artificial neural networks function and learn?
Q.3) 3.  What are the core components and architecture of a typical neural network?
Q.4) 4.  Describe the fundamental principles and applications of neural networks in AI.
Q.5) 5.  Break down the concept of neural networks and their purpose in machine learning.


- Reteriving the Documents for each and Every Question Generated by Multi-Query Generation Model

In [5]:
# --------------------------------------------------------------------------
# Reteriving the Documents for each and Every Question Generated by Multi-Query Generation Model
# --------------------------------------------------------------------------
# Purpose:
# For each reformulated query, retrieve the most semantically similar document
# chunks from the vector database. This step expands coverage across multiple
# formulations of the same information need.

retrieved_chunks = []  # To store retrieved chunks for each query

for question in generated_queries:
    print("\nProcessing Reformulated Query:", question)
    
    # Retrieve the top matching chunk for the current reformulated query
    retrieved_chunk = retrieve_most_similar_chunk(model, chunks, vector_store, question)
    
    retrieved_chunks.append(retrieved_chunk)

# Display the mapping between each reformulated query and its retrieved document chunk
print("\nRetrieved Chunks for Each Reformulated Query:")
for idx, question in enumerate(generated_queries):
    print(f"{idx+1}. {question} | Retrieved Chunk: {retrieved_chunks[idx]}")


Processing Reformulated Query: 1.  Please provide a detailed explanation of what neural networks are.
Calculated cosine similarity scores for each chunk:
Score = 0.2807  |  Chunk Index = 0
Score = 0.5821  |  Chunk Index = 1
Score = 0.6569  |  Chunk Index = 2
Score = 0.3293  |  Chunk Index = 3
Score = 0.0064  |  Chunk Index = 4
Score = 0.2673  |  Chunk Index = 5
Score = 0.4659  |  Chunk Index = 6
Score = 0.1862  |  Chunk Index = 7
Score = 0.0100  |  Chunk Index = 8
Score = 0.2516  |  Chunk Index = 9
Score = 0.1678  |  Chunk Index = 10
Score = 0.1663  |  Chunk Index = 11
Score = 0.0572  |  Chunk Index = 12

Processing Reformulated Query: 2.  How do artificial neural networks function and learn?
Calculated cosine similarity scores for each chunk:
Score = 0.4494  |  Chunk Index = 0
Score = 0.5834  |  Chunk Index = 1
Score = 0.5889  |  Chunk Index = 2
Score = 0.3403  |  Chunk Index = 3
Score = -0.0162  |  Chunk Index = 4
Score = 0.2862  |  Chunk Index = 5
Score = 0.5472  |  Chunk Index = 6

- Generation

In [6]:
# --------------------------------------------------------------------------
# Generation
# --------------------------------------------------------------------------
# Purpose:
# Use the retrieved context to synthesize a factual, concise, and research-oriented
# response. The LLM is instructed to maintain academic tone, factual accuracy,
# and avoid hallucination or unsupported reasoning.

agent_prompt = """
    You are an intelligent research assistant capable of grounded reasoning.
    Your task is to generate a concise, factual, and contextually accurate answer
    based on the retrieved context provided below.
    
    -------------------------------
    Context:
    {retrieved_chunk}
    -------------------------------
    
    Question:
    {user_query}

    Instructions:
    1. Use the provided retrieved context to answer the user query.
    2. If the context does not contain enough information, clearly state that.
    3. Avoid hallucination or assumptions beyond the provided context.
    4. Respond in clear, structured language suitable for a research explanation.
    5. Do not format the answer in markdown — respond in plain text only.
    6. Summarize the relevant information from the retrieved context before answering.
    7. Enhance the clarity, coherence, and academic tone of the response to ensure it reads like a refined summary, not raw text output.
    """

# Create the prompt and response generation chain
prompt = ChatPromptTemplate.from_messages([
    ("system", agent_prompt),
    ("user", "{user_query}")
])

# Build the reasoning-generation chain
chain = (prompt | llm | StrOutputParser())

# Invoke the chain to generate a final answer using the retrieved context
response = chain.invoke({
    "retrieved_chunk": retrieved_chunks,
    "user_query": query
})

print("\nGenerated Response:\n")
print(response)


Generated Response:

Neural networks are computational models inspired by the structure of the human brain. They constitute the fundamental basis of deep learning.


## **RAG Fusion**

**RAG Fusion** (also referred to as **Query Fusion** or **Multi-Query RAG**) is an advanced enhancement to the **Retrieval-Augmented Generation (RAG)** architecture.
It aims to improve the **quality, diversity, and completeness** of information retrieved from external knowledge sources (e.g., vector databases, document stores, or search indexes) before generating the final answer through a language model.

The core principle of RAG Fusion lies in **combining results from multiple semantically reformulated queries**, thereby creating a more **context-rich and diverse retrieval set**.

---

## **Motivation**

Conventional RAG pipelines depend on a **single user query** for document retrieval. This approach introduces several limitations:

* It often fails to capture all **semantic nuances** of the user’s intent.
* It may **miss relevant results** due to rigid phrasing or keyword sensitivity.
* It risks producing **incomplete or biased** contextual knowledge for the generator.

**RAG Fusion** mitigates these issues by generating **multiple query reformulations**, retrieving information for each, and then intelligently **fusing** the retrieved results into a single, high-quality document set.

---

## **Mechanism**

### **Step-1: Multi-Query Generation**

The system uses an LLM to create several **semantic variations** of the user’s question, such as:

* “How do electric vehicles affect the environment?”
* “What are the ecological advantages and drawbacks of EVs?”
* “Environmental footprint analysis of electric cars.”
* “Are EVs more sustainable than gasoline vehicles?”

This ensures **semantic diversity**, enabling the retrievers to explore multiple interpretive angles of the same intent.

---

### **Step-2: Retriever for Each Query**

Each reformulated query is processed by an **independent retriever**, such as a vector database or BM25 search model.
This produces multiple **ranked document lists**, for example:

* (q₁): [Doc A, Doc B, Doc C]
* (q₂): [Doc B, Doc D, Doc E]
* (q₃): [Doc F, Doc A, Doc D]

---

### **Step-3: Apply RRF on the Retrieved Documents from Multiple Retrievers**

All these ranked lists are merged and re-ranked during the **Fusion Phase**.
A widely used approach for this is **Reciprocal Rank Fusion (RRF)**, which aggregates document importance across queries.

#### **Details on How RRF Works**

1. **Formula:**
$$
\text{RRF}(d) = \sum_{q_i \in Q} \frac{1}{k + \text{rank}_{q_i}(d)}
$$
2. **Explanation:**
   Using ($\text{RRF}(d) = \sum_{q_i \in Q} \frac{1}{k + \text{rank}_{q_i}(d)}$), RRF assigns **greater weight to higher-ranked documents** (i.e., those appearing near the top).
   This ensures that documents ranked highly by multiple retrievers are **favoured** in the final ranking.
3. **Parameter:**
   The constant ( k ) (typically ≈ 60) smooths score differences, preventing outliers from dominating.

---

### **4. Context Construction**

The top-ranked documents from the fused list are combined, summarized, or concatenated to form the **context window** that will be supplied to the generator.

---

### **5. Generation**

The LLM uses this fused context to generate a **more accurate, evidence-based, and complete** final answer.

---

## **Algorithm — Reciprocal Rank Fusion (RRF)**

The **Reciprocal Rank Fusion** method is central to the Fusion Phase. It produces a unified ranking that balances **relevance**, **diversity**, and **agreement** across retrievers.

$$
\text{RRF}(d) = \sum_{q_i \in Q} \frac{1}{k + \text{rank}_{q_i}(d)}
$$

Where:
* ( Q ) — the set of reformulated queries.
* ( $\text{rank}_{q_i}(d)$ ) — the position of document ( d ) within the results for query ( ${q}_i$ ).
* ( k ) — a dampening constant controlling the effect of rank distance.

Documents that appear **frequently and early** in multiple ranked lists achieve higher cumulative scores and thus are placed at the top of the final ranking.

---

## **Practical Illustration**

Consider a research assistant chatbot retrieving academic papers:

* **Without RAG Fusion:** retrieves only a few results for “machine learning security.”
* **With RAG Fusion:** generates variants such as

  * “Adversarial attacks on neural networks”
  * “Robustness in machine learning models”
  * “Security vulnerabilities in AI systems”

The fused retrieval covers broader ground, minimizes bias, and strengthens factual grounding.

---

## **Summary Table**

| **Stage**                       | **Purpose**                               |
| ------------------------------- | ----------------------------------------- |
| Step 1 — Multi-Query Generation | Capture diverse semantic meanings         |
| Step 2 — Independent Retrieval  | Gather results per query                  |
| Step 3 — RRF Fusion             | Merge and re-rank by aggregated relevance |
| Step 4 — Context Building       | Assemble top documents for LLM input      |
| Step 5 — Generation             | Produce grounded, comprehensive output    |

---

## **Key Takeaways**

* **RAG Fusion = Multi-Query Generation + Retriever Diversity + RRF-Based Fusion.**
* Enhances **recall** by expanding semantic reach.
* Increases **accuracy** by integrating overlapping evidence.
* Reduces **hallucination** through multi-source grounding.
* Enables **contextual completeness** vital for high-precision knowledge generation.

### RAG-Fusion Implementation

- Loading and chunking the Documents

In [7]:
# Import necessary class
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Create a text splitter with defined chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 25,    # Each chunk will contain around 10 tokens
    chunk_overlap = 3   # 3 tokens of overlap between consecutive chunks
)

def load_documents():
    """
    Loads a small, diverse set of documents to simulate a knowledge base
    for Retrieval-Augmented Generation (RAG) experiments.

    Each document represents a single knowledge unit that will later be
    chunked, embedded, and indexed for similarity-based retrieval.

    Returns
    -------
    documents : list of langchain_core.documents.Document
        A list containing 10 textual documents across multiple domains.
    """

    # Define a diverse mini knowledge base covering various topics

    documents = [
        Document(page_content="Machine learning allows systems to automatically improve from experience without being explicitly programmed."),
        Document(page_content="Deep learning, a subset of machine learning, uses neural networks to model complex patterns in data."),
        Document(page_content="Neural networks are inspired by the structure of the human brain and form the foundation of deep learning."),
        Document(page_content="Python is a leading language for machine learning due to libraries like TensorFlow, PyTorch, NumPy, and Pandas."),
        Document(page_content="Data scientists use Python extensively for tasks like preprocessing, visualization, and training neural models."),
        Document(page_content="Artificial intelligence combines subfields like machine learning, deep learning, and natural language processing to mimic human intelligence."),
        Document(page_content="OpenAI’s GPT models leverage large datasets and transformer-based neural networks for human-like text generation."),
        Document(page_content="Retrieval-Augmented Generation (RAG) enhances language models by combining document retrieval with generation for factual accuracy."),
        Document(page_content="Large language models, such as GPT and Gemini, are trained using massive text corpora to understand and generate human-like responses."),
        Document(page_content="In modern AI pipelines, RAG systems use retrievers like BM25 and dense embeddings to fetch relevant information before generating answers.")
    ]


    # Log summary information for verification
    print(f"Total documents loaded: {len(documents)}")
    print("Displaying sample documents:")
    for i, doc in enumerate(documents[:3]):
        print(f"Document {i+1}: {doc.page_content[:100]}...")

    return documents
    
documents = load_documents()

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

# Display the created chunks
print(f"\nTotal chunks created: {len(chunks)}")
print("Displaying first 2 sample chunks:")
for i, chunk in enumerate(chunks[:2]):
    print(f"Chunk {i+1}: {chunk.page_content}")

Total documents loaded: 10
Displaying sample documents:
Document 1: Machine learning allows systems to automatically improve from experience without being explicitly pr...
Document 2: Deep learning, a subset of machine learning, uses neural networks to model complex patterns in data....
Document 3: Neural networks are inspired by the structure of the human brain and form the foundation of deep lea...

Total chunks created: 13
Displaying first 2 sample chunks:
Chunk 1: Machine learning allows systems to automatically improve from experience without being explicitly programmed.
Chunk 2: Deep learning, a subset of machine learning, uses neural networks to model complex patterns in data.


- Indexing the Documents

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer("all-MiniLM-L6-v2")
# Initialize an empty vector store to hold embeddings for each chunk
vector_store = []

# Convert each text chunk into a dense vector representation (embedding)
for idx, chunk in enumerate(chunks):
    text_content = chunk.page_content
    embedding_vector = model.encode(text_content)   # Convert text → vector
    vector_store.append(embedding_vector)

# Display the first 5 dimensions of first two chunk embeddings (for inspection)
[vector_store[0][:5], vector_store[1][:5]]

[array([-0.00684425,  0.0059964 ,  0.08135018,  0.06403239,  0.02934441],
       dtype=float32),
 array([-0.09046526, -0.02000159,  0.05460881, -0.00526585, -0.03021288],
       dtype=float32)]

- Creating Multiple-Query

In [9]:
# Function to calculate cosine similarity between two vectors
def calculate_cosine_similarity_score_vect(v1, v2):
    dot_product = np.dot(v1, v2)
    v1_magnitude = np.linalg.norm(v1)
    v2_magnitude = np.linalg.norm(v2)
    cosine_similarity = dot_product / (v1_magnitude * v2_magnitude)
    return cosine_similarity

def retrieve_most_similar_chunk(model, chunks, vector_store, query_text):
    """
    Given a query, finds the most semantically similar chunk.
    Steps:
    1. Encode the query text into a vector.
    2. Compute cosine similarity with every chunk vector.
    3. Return the chunk with the highest similarity score.
    """
    encoded_query = model.encode(query_text)

    similarity_scores = []
    indexed_scores = []

    # Calculate similarity for each stored embedding
    for idx, chunk_vector in enumerate(vector_store):
        score = calculate_cosine_similarity_score_vect(encoded_query, chunk_vector)
        similarity_scores.append(score)
        indexed_scores.append((score, idx))

    # Log all similarity values for reference
    print("Calculated cosine similarity scores for each chunk:")
    for score, idx in indexed_scores:
        print(f"Score = {score:.4f}  |  Chunk Index = {idx}")

    # Retrieve the chunk with the maximum similarity score
    best_match_index = np.argmax(similarity_scores)
    best_chunk = chunks[best_match_index].page_content
    return best_chunk

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAI

# --------------------------------------------------------------------------
# Multi-Query Generation Model
# --------------------------------------------------------------------------
# Purpose:
# Generate multiple semantically diverse reformulations of the user's query.
# Each reformulated query represents a different perspective to improve
# document retrieval recall and reduce semantic bias in vector similarity search.

# Define the prompt template for the LLM
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 
    """
    You are an advanced language model. Your task is to generate five diverse reformulations 
    of the given user query to enhance document retrieval from a vector database.
    The objective is to capture multiple semantic perspectives of the original query,
    thereby mitigating the limitations of distance-based similarity search.
    Each reformulated query should be presented on a new line.
    """),
    ("user", "{user_query}")
])

# Initialize the generative LLM (Gemini 2.5-Flash)
llm = GoogleGenerativeAI(model="gemini-2.5-flash")

# Build the Multi-Query Generation Chain
# This chain takes the user query, sends it to the LLM, parses the output,
# and splits the generated reformulations line by line.
multi_query_chain = (prompt_template | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Example user query
query = "What are neural networks?"

# Generate multiple diverse reformulations for the given query
generated_queries = multi_query_chain.invoke(query)

print("Original Query:", query)
print("\nGenerated Queries by Multi-Query Generation Model:")
for idx, question in enumerate(generated_queries, start=1):
    print(f"Q.{idx}) {question}")

Original Query: What are neural networks?

Generated Queries by Multi-Query Generation Model:
Q.1) Define neural networks.
Q.2) Explain the fundamental concept of neural networks.
Q.3) What are the core components and architecture of a neural network?
Q.4) What is the purpose and functionality of neural networks in machine learning?
Q.5) How do neural networks process information and learn?


- Reteriving the Documents for each and every Query Generated by Multi-Query Generation Model

In [11]:
# --------------------------------------------------------------------------
# Reteriving the Documents for each and Every Question Generated by Multi-Query Generation Model
# --------------------------------------------------------------------------
# Purpose:
# For each reformulated query, retrieve the most semantically similar document
# chunks from the vector database. This step expands coverage across multiple
# formulations of the same information need.

retrieved_chunks = []  # To store retrieved chunks for each query

for question in generated_queries:
    print("\nProcessing Reformulated Query:", question)
    
    # Retrieve the top matching chunk for the current reformulated query
    retrieved_chunk = retrieve_most_similar_chunk(model, chunks, vector_store, question)
    
    retrieved_chunks.append(retrieved_chunk)

# Display the mapping between each reformulated query and its retrieved document chunk
print("\nRetrieved Chunks for Each Reformulated Query:")
for idx, question in enumerate(generated_queries):
    print(f"{idx+1}. {question} | Retrieved Chunk: {retrieved_chunks[idx]}")


Processing Reformulated Query: Define neural networks.
Calculated cosine similarity scores for each chunk:
Score = 0.2998  |  Chunk Index = 0
Score = 0.5618  |  Chunk Index = 1
Score = 0.6448  |  Chunk Index = 2
Score = 0.2976  |  Chunk Index = 3
Score = 0.0188  |  Chunk Index = 4
Score = 0.2329  |  Chunk Index = 5
Score = 0.4043  |  Chunk Index = 6
Score = 0.2197  |  Chunk Index = 7
Score = 0.0109  |  Chunk Index = 8
Score = 0.2336  |  Chunk Index = 9
Score = 0.1503  |  Chunk Index = 10
Score = 0.1458  |  Chunk Index = 11
Score = 0.0218  |  Chunk Index = 12

Processing Reformulated Query: Explain the fundamental concept of neural networks.
Calculated cosine similarity scores for each chunk:
Score = 0.3630  |  Chunk Index = 0
Score = 0.6214  |  Chunk Index = 1
Score = 0.7047  |  Chunk Index = 2
Score = 0.3254  |  Chunk Index = 3
Score = -0.0306  |  Chunk Index = 4
Score = 0.2614  |  Chunk Index = 5
Score = 0.4450  |  Chunk Index = 6
Score = 0.2267  |  Chunk Index = 7
Score = 0.0608  |

* Compute the Reciprocal Rank Fusion (RRF) scores for the retrieved documents

In [12]:
# ==============================================================
# Reciprocal Rank Fusion (RRF) — Detailed Implementation
# ==============================================================

def reciprocal_rank_fusion(retrieved_docs, k=60):
    """
    Performs Reciprocal Rank Fusion (RRF) on multiple retrieved document lists.

    Formula:
        RRF_score(doc) = Σ (1 / (k + rank))

    Description:
        - Documents appearing at higher ranks (lower rank numbers) receive higher scores.
        - If a document is ranked highly by multiple retrievers, its total RRF score increases.
        - This method ensures that the final ranking reflects consensus among multiple retrieval sources.

    Parameters:
        retrieved_docs : list
            Combined ranked list of documents retrieved from multiple retrievers or query reformulations.
        k : int
            Constant to smooth out rank influence and prevent dominance of a single retriever. Default = 60.

    Returns:
        list of tuples
            Returns a list of (document, RRF_score) sorted by descending score.
    """

    rrf_scores = {}

    print("\nApplying Reciprocal Rank Fusion on Retrieved Documents")
    print("--------------------------------------------------------------")
    print(f"Number of retrieved document chunks: {len(retrieved_docs)}")
    print(f"Constant 'k' used for rank smoothing: {k}\n")

    # Compute the RRF score for each document
    for rank_position, document in enumerate(retrieved_docs):
        rrf_scores[document] = rrf_scores.get(document, 0) + (1 / (k + rank_position))

    # Sort documents based on their final aggregated RRF score
    sorted_rankings = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    print("Reciprocal Rank Fusion completed successfully.\n")
    return sorted_rankings


# ==============================================================
# Fusing Multi-Retriever Results
# ==============================================================

ranked_results = reciprocal_rank_fusion(retrieved_chunks)

final_selected_docs = []

print("\nRRF Scoring Analysis")
print("=" * 70)
print(f"{'Rank':<6}{'Document Chunk (Truncated)':<55}{'RRF Score'}")
print("-" * 70)

# Display top-ranked documents with their corresponding RRF scores
for idx, (doc_chunk, rrf_score) in enumerate(ranked_results):
    truncated_doc = (doc_chunk[:50] + "...") if len(doc_chunk) > 50 else doc_chunk
    final_selected_docs.append(doc_chunk)
    print(f"{idx+1:<6}{truncated_doc:<55}{rrf_score:.5f}")


# ==============================================================
# Selecting Top-K Documents for Final Context
# ==============================================================

top_k = 2  # Adjust this based on LLM context size or token limitations

# Merge top-ranked documents into a single contextual input for the LLM
final_context = "\n\n".join(final_selected_docs[:top_k])

print("\n\nFinal Context Construction")
print("=" * 70)
print(f"Selected Top-{top_k} Documents after Fusion:")
print("-" * 70)
for i, doc in enumerate(final_selected_docs[:top_k]):
    print(f"[{i+1}] {doc[:200]}{'...' if len(doc) > 200 else ''}")


# ==============================================================
# Comparative Analysis — Before vs After RAG Fusion
# ==============================================================

print("\n\nComparative Context Overview")
print("=" * 70)
print("Before RAG-Fusion → Context directly passed to LLM (Raw Retrievals):")
print("-" * 70)
for i, chunk in enumerate(retrieved_chunks):
    print(f"{i+1}. {chunk[:150]}{'...' if len(chunk) > 150 else ''}")

print("\nAfter RAG-Fusion → Context passed to LLM (Fused & Optimized):")
print("-" * 70)
print(final_context[:800] + ("..." if len(final_context) > 800 else ""))
print("-" * 70)

print("\nObservation:")
print("→ RAG-Fusion consolidates semantically similar yet complementary chunks,")
print("  reducing redundancy and ensuring stronger contextual grounding for the LLM.")
print("→ Higher-ranked documents, based on agreement across retrievers,")
print("  are prioritized to enhance factual accuracy and relevance.\n")

print("Context is now ready to be passed into the LLM for final response generation.")
print("=" * 70)


Applying Reciprocal Rank Fusion on Retrieved Documents
--------------------------------------------------------------
Number of retrieved document chunks: 5
Constant 'k' used for rank smoothing: 60

Reciprocal Rank Fusion completed successfully.


RRF Scoring Analysis
Rank  Document Chunk (Truncated)                             RRF Score
----------------------------------------------------------------------
1     Neural networks are inspired by the structure of t...  0.04919
2     Deep learning, a subset of machine learning, uses ...  0.03150


Final Context Construction
Selected Top-2 Documents after Fusion:
----------------------------------------------------------------------
[1] Neural networks are inspired by the structure of the human brain and form the foundation of deep learning.
[2] Deep learning, a subset of machine learning, uses neural networks to model complex patterns in data.


Comparative Context Overview
Before RAG-Fusion → Context directly passed to LLM (Raw Retrieva

- Generation

In [13]:
# --------------------------------------------------------------------------
# Generation
# --------------------------------------------------------------------------
# Purpose:
# Use the retrieved context to synthesize a factual, concise, and research-oriented
# response. The LLM is instructed to maintain academic tone, factual accuracy,
# and avoid hallucination or unsupported reasoning.

agent_prompt = """
    You are an intelligent research assistant capable of grounded reasoning.
    Your task is to generate a concise, factual, and contextually accurate answer
    based on the retrieved context provided below.
    
    -------------------------------
    Context:
    {retrieved_chunk}
    -------------------------------
    
    Question:
    {user_query}

    Instructions:
    1. Use the provided retrieved context to answer the user query.
    2. If the context does not contain enough information, clearly state that.
    3. Avoid hallucination or assumptions beyond the provided context.
    4. Respond in clear, structured language suitable for a research explanation.
    5. Do not format the answer in markdown — respond in plain text only.
    6. Summarize the relevant information from the retrieved context before answering.
    7. Enhance the clarity, coherence, and academic tone of the response to ensure it reads like a refined summary, not raw text output.
    """

# Create the prompt and response generation chain
prompt = ChatPromptTemplate.from_messages([
    ("system", agent_prompt),
    ("user", "{user_query}")
])

# Build the reasoning-generation chain
chain = (prompt | llm | StrOutputParser())

# Invoke the chain to generate a final answer using the final retrieved context
response = chain.invoke({
    "retrieved_chunk": final_context,
    "user_query": query
})

print("\nGenerated Response:\n")
print(response)


Generated Response:

Neural networks are computational models inspired by the structure of the human brain. They constitute the fundamental basis of deep learning, a specialized area within machine learning, and are employed to model intricate patterns found within data.
